# Investment Strategies Testing Framework

This notebook demonstrates how to use the comprehensive framework for testing different investment strategies on historical stock data.

## Features:
- Fetch historical stock data from Yahoo Finance
- Implement various trading strategies
- Backtest strategies with realistic trading costs
- Compare strategy performance
- Visualize results with comprehensive charts

## 1. Setup and Imports

In [ ]:
# Import necessary libraries
import sys
import os

# Add src directory to path
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Import our custom modules
from data_fetcher import DataFetcher
from backtest_engine import BacktestEngine
from visualizer import PerformanceVisualizer
from strategies import (
    BuyAndHoldStrategy, 
    MovingAverageCrossoverStrategy, 
    MomentumStrategy, 
    MeanReversionStrategy
)

# Import configuration
import config

print("All imports successful!")

## 2. Configuration and Data Setup

In [ ]:
# Set up configuration
SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA']
START_DATE = datetime.now() - timedelta(days=5*365)  # 5 years ago
END_DATE = datetime.now()
INITIAL_CAPITAL = 100000  # $100,000
BENCHMARK = 'SPY'

print(f"Testing period: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}")
print(f"Symbols to trade: {SYMBOLS}")
print(f"Initial capital: ${INITIAL_CAPITAL:,}")

In [ ]:
# Initialize data fetcher and backtest engine
data_fetcher = DataFetcher(data_dir='../data')
backtest_engine = BacktestEngine(data_fetcher)
visualizer = PerformanceVisualizer()

print("Framework initialized!")

## 3. Fetch and Explore Data

In [ ]:
# Fetch sample data for exploration
print("Fetching sample data for AAPL...")
sample_data = data_fetcher.get_stock_data('AAPL', START_DATE, END_DATE)

print(f"Data shape: {sample_data.shape}")
print(f"Date range: {sample_data.index[0]} to {sample_data.index[-1]}")
print("\nFirst few rows:")
sample_data.head()

In [ ]:
# Visualize sample data
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Price chart
axes[0].plot(sample_data.index, sample_data['Close'], linewidth=2, label='AAPL Close Price')
axes[0].set_title('AAPL Price History (5 Years)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume chart
axes[1].bar(sample_data.index, sample_data['Volume'], alpha=0.7, width=1)
axes[1].set_title('AAPL Volume History', fontsize=12)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Basic statistics
print("Basic Statistics for AAPL:")
print(sample_data['Close'].describe())

## 4. Initialize Trading Strategies

In [ ]:
# Initialize different strategies
strategies = [
    BuyAndHoldStrategy(initial_capital=INITIAL_CAPITAL),
    MovingAverageCrossoverStrategy(
        short_window=20, 
        long_window=50, 
        initial_capital=INITIAL_CAPITAL
    ),
    MomentumStrategy(
        rsi_period=14, 
        momentum_period=10,
        initial_capital=INITIAL_CAPITAL
    ),
    MeanReversionStrategy(
        bb_period=20, 
        bb_std=2.0,
        initial_capital=INITIAL_CAPITAL
    )
]

print("Strategies initialized:")
for strategy in strategies:
    print(f"- {strategy.name}")

## 5. Run Backtests

In [ ]:
# Run backtests for all strategies
results = []

print("Running backtests...\n")
for i, strategy in enumerate(strategies, 1):
    print(f"[{i}/{len(strategies)}] Running backtest for {strategy.name}...")
    
    try:
        result = backtest_engine.run_backtest(
            strategy=strategy,
            symbols=SYMBOLS,
            start_date=START_DATE,
            end_date=END_DATE,
            benchmark_symbol=BENCHMARK
        )
        results.append(result)
        print(f"✓ Completed: Total Return = {result['total_return']:.2f}%\n")
        
    except Exception as e:
        print(f"✗ Failed: {str(e)}\n")
        continue

print(f"Completed {len(results)} backtests successfully!")

## 6. Performance Summary

In [ ]:
# Create performance comparison table
if results:
    comparison_df = backtest_engine.compare_strategies(results)
    
    print("Strategy Performance Comparison:")
    print("=" * 80)
    
    # Display formatted results
    display_df = comparison_df.copy()
    
    # Format columns for better readability
    for col in ['Total_Return_%', 'Annualized_Return_%', 'Volatility_%', 'Max_Drawdown_%', 'Win_Rate_%']:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.2f}%")
    
    if 'Sharpe_Ratio' in display_df.columns:
        display_df['Sharpe_Ratio'] = display_df['Sharpe_Ratio'].apply(lambda x: f"{x:.3f}")
    
    if 'Final_Value' in display_df.columns:
        display_df['Final_Value'] = display_df['Final_Value'].apply(lambda x: f"${x:,.0f}")
    
    display(display_df)
else:
    print("No successful backtest results to display.")

## 7. Detailed Analysis for Best Performing Strategy

In [ ]:
if results:
    # Find best performing strategy by total return
    best_strategy = max(results, key=lambda x: x['total_return'])
    
    print(f"Best Performing Strategy: {best_strategy['strategy_name']}")
    print(f"Total Return: {best_strategy['total_return']:.2f}%")
    print(f"Final Portfolio Value: ${best_strategy['final_portfolio_value']:,.2f}")
    
    # Display detailed metrics
    print("\nDetailed Performance Metrics:")
    metrics = best_strategy['performance_metrics']
    for key, value in metrics.items():
        if 'Pct' in key or 'Rate' in key:
            print(f"{key.replace('_', ' ').replace('Pct', '(%)')}: {value:.2f}%")
        else:
            print(f"{key.replace('_', ' ')}: {value:.3f}")

In [ ]:
# Plot detailed performance for best strategy
if results:
    visualizer.plot_portfolio_performance(best_strategy)
    visualizer.plot_drawdown_analysis(best_strategy)
    visualizer.plot_returns_distribution(best_strategy)

## 8. Strategy Comparison Visualizations

In [ ]:
# Compare all strategies
if len(results) > 1:
    visualizer.plot_strategy_comparison(results)
else:
    print("Need at least 2 successful strategies for comparison.")

## 9. Trade Analysis

In [ ]:
# Analyze trades for the best strategy
if results and not best_strategy['trades'].empty:
    print(f"Trade Analysis for {best_strategy['strategy_name']}:")
    print(f"Total number of trades: {len(best_strategy['trades'])}")
    
    trades_df = best_strategy['trades']
    
    # Trade statistics
    buy_trades = len(trades_df[trades_df['Signal'] == 'BUY'])
    sell_trades = len(trades_df[trades_df['Signal'] == 'SELL'])
    
    print(f"Buy trades: {buy_trades}")
    print(f"Sell trades: {sell_trades}")
    
    # Average trade size
    avg_trade_size = trades_df['Value'].mean()
    print(f"Average trade size: ${avg_trade_size:,.2f}")
    
    # Total commission paid
    total_commission = trades_df['Commission'].sum()
    print(f"Total commission paid: ${total_commission:.2f}")
    
    # Plot trade analysis
    visualizer.plot_trades_analysis(best_strategy)
    
    # Show first few trades
    print("\nFirst 10 Trades:")
    display(trades_df.head(10))
else:
    print("No trades to analyze.")

## 10. Interactive Dashboard (Optional)

In [ ]:
# Create interactive dashboard for best strategy
if results:
    try:
        print("Creating interactive dashboard...")
        visualizer.create_interactive_dashboard(best_strategy)
    except ImportError:
        print("Plotly not available for interactive dashboard. Install with: pip install plotly")
    except Exception as e:
        print(f"Error creating dashboard: {e}")

## 11. Save Results

In [ ]:
# Save results to CSV files
if results:
    results_dir = '../results'
    os.makedirs(results_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save comparison summary
    comparison_file = f'{results_dir}/strategy_comparison_{timestamp}.csv'
    comparison_df.to_csv(comparison_file)
    print(f"Strategy comparison saved to: {comparison_file}")
    
    # Save detailed results for each strategy
    for result in results:
        strategy_name = result['strategy_name'].replace(' ', '_').replace('(', '').replace(')', '')
        
        # Portfolio history
        portfolio_file = f'{results_dir}/{strategy_name}_portfolio_{timestamp}.csv'
        result['portfolio_history'].to_csv(portfolio_file)
        
        # Trades (if any)
        if not result['trades'].empty:
            trades_file = f'{results_dir}/{strategy_name}_trades_{timestamp}.csv'
            result['trades'].to_csv(trades_file, index=False)
        
        print(f"Results for {result['strategy_name']} saved to {results_dir}/")
    
    print("\nAll results saved successfully!")
else:
    print("No results to save.")

## 12. Next Steps and Customization

This framework provides a solid foundation for testing investment strategies. Here are some ways to extend it:

### Custom Strategies
- Create your own strategy by inheriting from `BaseStrategy`
- Implement the `generate_signals()` method
- Use technical indicators or machine learning models

### Advanced Features
- Add more sophisticated risk management rules
- Implement portfolio optimization techniques
- Add support for options and derivatives
- Include fundamental analysis data

### Performance Enhancements
- Parallel processing for multiple backtests
- Database integration for faster data access
- Real-time data feeds

### Risk Management
- Position sizing based on volatility
- Stop-loss and take-profit orders
- Portfolio-level risk controls
- VaR (Value at Risk) calculations

### Validation
- Walk-forward analysis
- Cross-validation techniques
- Out-of-sample testing
- Monte Carlo simulations

In [ ]:
# Framework summary
print("🎉 Investment Strategies Testing Framework Demo Complete!")
print("\n📊 Framework Components:")
print("- Data Fetcher: Historical stock data with caching")
print("- Strategy Library: Buy & Hold, MA Crossover, Momentum, Mean Reversion")
print("- Backtest Engine: Realistic trading simulation with costs")
print("- Performance Analytics: Comprehensive metrics calculation")
print("- Visualization Tools: Static and interactive charts")

print("\n🚀 Ready to test your own strategies!")
print("\nHappy investing! 📈")